# Recherche d'information sur le corpus CISI — Système 2 : recherche dense et re-ranking

**Auteurs :** Melissa Boko et Amadou Bah — projet de Traitement Automatique des Langues, INSA Rennes

**Objectif :** améliorer le système 1 (TF-IDF) en comparant les requêtes et les documents selon leur **sens**, et non plus seulement selon les mots qu'ils partagent.

**Approche en deux étapes**, classique dans les moteurs de recherche modernes :

1. **Recherche dense (bi-encodeur)** : `all-MiniLM-L12-v2` transforme chaque document et chaque requête en vecteur (*embedding*), **séparément**. Les documents sont encodés une seule fois, puis on garde les documents les plus proches de la requête (similarité cosinus). Cette étape est rapide, mais un peu approximative.
2. **Re-ranking (cross-encodeur)** : `ms-marco-MiniLM-L12-v2` lit la requête et chaque document candidat **ensemble**, ce qui lui permet de juger beaucoup plus finement leur pertinence. Cette étape est plus coûteuse, c'est pourquoi on ne l'applique qu'aux meilleurs candidats de l'étape 1.

Seuls les documents dont le score de re-ranking dépasse un seuil sont renvoyés.

*Exécution recommandée sur GPU (Colab : Exécution > Modifier le type d'exécution > GPU T4).*

## 0. Installation et configuration

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util

# Fichiers d'entrée (à placer dans /content sous Colab)
CORPUS_PATH = "/content/CISI.ALLnettoye"
QRY_PATH = "/content/CISI.QRY"
REL_PATH = "/content/CISI_dev.REL"
EVAL_SCRIPT_PATH = "/content/eval.pl"

# Fichier de résultats produit par ce système
RUN_PATH = "/content/run_dense_rerank.REL"

# Paramètres
TOP_K_CANDIDATS = 45      # nombre de candidats transmis au cross-encodeur
SEUIL_PERTINENCE = -4.5   # score minimal du cross-encodeur pour renvoyer un document

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

## 1. Chargement des modèles

In [ ]:
bi_encodeur = SentenceTransformer("all-MiniLM-L12-v2", device=device)
cross_encodeur = CrossEncoder("cross-encoder/ms-marco-MiniLM-L12-v2", device=device)

## 2. Parsing du corpus et des requêtes

In [ ]:
def parser_corpus(chemin_fichier):
    documents = {}
    numero, contenu = None, ""
    with open(chemin_fichier, "r", encoding="utf-8", errors="ignore") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(".I"):
                if numero is not None:
                    documents[numero] = contenu.strip()
                numero, contenu = int(ligne.split()[1]), ""
            else:
                contenu += " " + ligne
    if numero is not None:
        documents[numero] = contenu.strip()
    return documents


def parser_requetes(chemin_fichier, sections_a_garder=(".T", ".W")):
    """Ne conserve que le titre (.T) et le texte (.W) de chaque requête, pas les auteurs (.A) ni la source (.B)."""
    requetes = {}
    numero, contenu, capture = None, "", False
    with open(chemin_fichier, "r", encoding="utf-8", errors="ignore") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith(".I"):
                if numero is not None:
                    requetes[numero] = contenu.strip()
                numero, contenu, capture = int(ligne.split()[1]), "", False
            elif ligne in {".T", ".A", ".W", ".B"}:
                capture = ligne in sections_a_garder
            elif capture:
                contenu += " " + ligne
    if numero is not None:
        requetes[numero] = contenu.strip()
    return requetes


documents = parser_corpus(CORPUS_PATH)
requetes = parser_requetes(QRY_PATH)
print(f"{len(documents)} documents, {len(requetes)} requêtes")

## 3. Encodage du corpus

Chaque document est transformé une seule fois en vecteur. Ces vecteurs sont ensuite réutilisés pour toutes les requêtes.

In [ ]:
doc_ids = list(documents.keys())
doc_textes = [documents[i] for i in doc_ids]

embeddings_corpus = bi_encodeur.encode(doc_textes, convert_to_tensor=True, show_progress_bar=True)
print("Dimensions des embeddings :", tuple(embeddings_corpus.shape))

## 4. Recherche dense puis re-ranking

Pour chaque requête :
1. on l'encode et on calcule sa similarité cosinus avec tous les documents ;
2. on garde les `TOP_K_CANDIDATS` documents les plus proches ;
3. le cross-encodeur attribue un score de pertinence à chaque paire (requête, document candidat) ;
4. on garde les documents au-dessus du seuil, triés par score décroissant.

In [ ]:
def rechercher(texte_requete):
    # Étape 1 : recherche dense
    embedding_requete = bi_encodeur.encode(texte_requete, convert_to_tensor=True)
    similarites = util.cos_sim(embedding_requete, embeddings_corpus)[0]
    top = torch.topk(similarites, k=min(TOP_K_CANDIDATS, len(doc_ids))).indices
    candidats = [doc_ids[i] for i in top.tolist()]

    # Étape 2 : re-ranking par le cross-encodeur
    paires = [[texte_requete, documents[c]] for c in candidats]
    scores = cross_encodeur.predict(paires, show_progress_bar=False)

    retenus = [(c, float(s)) for c, s in zip(candidats, scores) if s > SEUIL_PERTINENCE]
    return sorted(retenus, key=lambda x: x[1], reverse=True)


resultats = {n: rechercher(texte) for n, texte in requetes.items()}
print(f"{len(resultats)} requêtes traitées")

## 5. Écriture des résultats et évaluation

In [ ]:
with open(RUN_PATH, "w", encoding="utf-8") as f:
    for n_requete in sorted(resultats):
        for n_doc, score in resultats[n_requete]:
            f.write(f"{n_requete}\t{n_doc}\t{score:.6f}\n")

print("Résultats écrits dans", RUN_PATH)

In [ ]:
!perl {EVAL_SCRIPT_PATH} {REL_PATH} {RUN_PATH}

## 6. Résultats et comparaison

<!-- Recopie ici les scores affichés par eval.pl pour les deux systèmes -->

| Métrique | Système 1 : TF-IDF | Système 2 : dense + re-ranking |
|---|---|---|
| MAP | [X] | [X] |
| Précision | [X] | [X] |
| Rappel | [X] | [X] |
| F-mesure | [X] | [X] |

**Analyse :** [Explique ici l'écart entre les deux systèmes. Par exemple : le système neuronal retrouve des documents pertinents qui n'emploient pas les mêmes mots que la requête, ce que le TF-IDF ne peut pas faire.]

**Limites et pistes :** les modèles utilisés ont été entraînés sur des données généralistes (MS MARCO), et non sur des articles scientifiques en sciences de l'information. Un ajustement du seuil par validation, ou une combinaison des scores TF-IDF et neuronaux (recherche hybride), pourraient encore améliorer les résultats.